# Pilot experiment — free Colab T4### If you have never used Google Colab, read this box. It is all you need.1. You are already looking at the notebook. Good.2. **Turn the GPU on.** Top menu: **Runtime → Change runtime type** →   *Hardware accelerator* = **T4 GPU** → **Save**.   Leave "High-RAM" **off**. You do not need it.3. **Run cells one at a time, in order.** Click a cell, then press   **Shift + Enter**. Wait until the spinner stops and a number appears in   `[ ]` on the left before starting the next one.4. Some cells take a long time (Step 7 is 1–2 hours, Step 9 is 3–4 hours).   That is expected. **Keep this browser tab open** — Colab disconnects a   tab left alone.5. If it disconnects anyway: reconnect, re-run Steps 2–5, then jump straight   back to whichever step you were on. **Nothing is lost** — Step 3 saves   everything to your Google Drive and the code skips work already done.**Total time: about 5–6 hours.** You can stop after Step 8 and continueanother day.---## What this notebook is, in plain languageThe full experiment is 114 training runs and takes **4–5 days** on a freeT4 — impossible here. This notebook runs **12 carefully chosen runs** thatanswer the one question that decides whether the full experiment is worthstarting at all.### The questionWe have two AI models:| Model | Did it ever see Azerbaijani while being built? ||---|---|| **xlm15** | **No** || **xlmr** | **Yes** |Our theory: xlm15 struggles with Azerbaijani mainly because it *chops thewords up badly* — not because it fundamentally cannot learn the language.So if we swap in a proper Azerbaijani word-chopper (a "tokenizer"), xlm15should improve **a lot**, while xlmr — which already chops Azerbaijanifine — should barely change.### The 2×2 that tests it|  | original chopper | swapped chopper ||---|---|---|| **xlm15** (never saw Azerbaijani) | should be weak | **should jump** || **xlmr** (already knows it) | already good | should stay ~the same |That diagonal pattern is the whole hypothesis. **12 runs = 2 models × 2choppers × 3 repeats.** If the pattern shows up here, the full experimentis very likely to work. If it does not, you have saved a week and learnedsomething real.> **Important, and not a footnote.** These results are a **pilot**, not> paper results. They use a shorter training budget (800 steps instead of> 2000) and 3 repeats instead of 5, and they are written to a separate> folder `results_pilot/` so they can never get mixed into the real> experiment's numbers. Do not put these numbers in the report.

---## Step 1 — Check the GPU is onIf this says "NO GPU", go to **Runtime → Change runtime type → T4 GPU → Save**,then run this cell again.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU FOUND: {p.name}, {p.total_memory/1e9:.1f} GB")
        if "T4" not in p.name:
            print(f"(This is a {p.name}, not a T4. That is fine - it will just be faster.)")
        print("\nAll good. Continue to Step 2.")
    else:
        print("NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.")
except ImportError:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.")

---## Step 2 — Upload the codeYou were sent a **.zip** file of the project. Run the cell below, click**Choose Files**, and pick that zip.*(Upload takes a minute or two. Wait for "project ready" to appear.)*

In [ ]:
import os, shutil, sys, zipfile
from pathlib import Path

WORK = Path("/content/project")

if (WORK / "configs" / "pilot_t4.yaml").exists():
    print("Code is already here - skipping the upload.")
else:
    from google.colab import files
    print("Click 'Choose Files' below and select the project .zip you were sent.")
    uploaded = files.upload()
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(WORK)

# A zip often contains one folder inside it; step into it if so.
if not (WORK / "configs" / "pilot_t4.yaml").exists():
    inner = [p for p in WORK.iterdir() if p.is_dir() and (p / "configs").exists()]
    if len(inner) == 1:
        WORK = inner[0]

assert (WORK / "configs" / "pilot_t4.yaml").exists(), (
    f"Could not find the project inside {WORK}. Is this the right zip?")

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("project ready:", WORK)

---## Step 3 — Connect Google Drive (this is what saves your work)Colab wipes everything when it disconnects. This step points the results andthe slow-to-build files at your Drive instead, so a disconnect costs you**nothing**.When you run it, a popup asks you to pick your Google account and click**Allow**. Say yes.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/az-tokenizer-pilot")

for sub in ("results_pilot", "artifacts", "figures_pilot"):
    (STORE / sub).mkdir(parents=True, exist_ok=True)
    local = Path.cwd() / sub
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        for item in local.iterdir():          # keep anything shipped in the zip
            if not (STORE / sub / item.name).exists():
                (shutil.copytree if item.is_dir() else shutil.copy2)(
                    item, STORE / sub / item.name)
        shutil.rmtree(local)
    local.symlink_to(STORE / sub, target_is_directory=True)

print("Your work will be saved to:", STORE)
print("free disk:", f"{shutil.disk_usage(Path.cwd()).free/1e9:.0f} GB")

---## Step 4 — Install a few small packagesAbout 1 minute. Colab already has almost everything; these are the fewextras the models need.> **A note for the record:** the full experiment must be installed from> `requirements.lock` (exact pinned versions). This pilot deliberately uses> Colab's own PyTorch instead — it avoids a 2.5 GB download and a confusing> runtime restart, and the code has been verified to run correctly on both> the pinned stack and a current one. Because the pilot is not reportable,> that trade is fine **here and only here**.

In [ ]:
!pip install -q sentencepiece sacremoses accelerate datasets 2>&1 | tail -2
print("done")

---## Step 5 — Self-check (about 2 minutes)This runs the project's own test suite. It is here because it catchesbroken setups **before** you spend hours training.You want to see something like `91 passed`. If you see failures, stop andsend the output back — do not continue.

In [ ]:
!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('config files OK - not modified')"
!python -m pytest tests/ -q -m "not slow" --no-header 2>&1 | tail -5

In [ ]:
# What the pilot is about to run: 12 runs, and no slow "Turkish" stage.
!python -m src.training.run_grid --config configs/pilot_t4.yaml --dry-run

---## Step 6 — Download and prepare the data (about 10 minutes)Downloads the Azerbaijani and Turkish datasets and splits them intotrain / validation / test.The important line in the output is `leakage: {...all zeros...}`. Thatproves no test example leaked into training — if it were not zero, everyresult afterwards would be meaningless.

In [ ]:
!python -m src.data.splits   --config configs/pilot_t4.yaml
!python -m src.data.scramble --config configs/pilot_t4.yaml

In [ ]:
import json
s = json.load(open("results_pilot/splits.json", encoding="utf-8"))
print("Azerbaijani examples -> train:", s["az"]["train"],
      "| validation:", s["az"]["val"], "| test:", s["az"]["test"])
print("labels:", s["az"]["label_map"])
print("duplicates removed:", s["az"]["dedup"]["duplicate_pct"], "%")
print("leakage check:", s["leakage_check"])
assert not any(s["leakage_check"].values()), "STOP: test data leaked into training"
print("\nClean. Continue to Step 7.")

---## Step 7 — Build the swapped tokenizer  ⏳ **1–2 hours**This is the slow part. It rebuilds each model's word-embedding table for theAzerbaijani chopper, one word at a time — about 25,000 words per model, onCPU.**Start this and go do something else.** Keep the tab open. When it finishesyou will see two `artifacts/transplanted__...` folders.If you get disconnected during this step, just re-run Steps 2–5 then thiscell again — finished pieces are saved on your Drive and are reused.

In [ ]:
import time
t0 = time.time()
!python -m src.transplant.build --config configs/pilot_t4.yaml
!python -m src.transplant.build_rescale_variant --config configs/pilot_t4.yaml
print(f"\nTOTAL BUILD TIME: {(time.time()-t0)/60:.0f} minutes")

In [ ]:
from pathlib import Path
built = sorted(p.name for p in Path("artifacts").glob("transplanted__*"))
print("Built:", *built, sep="\n  ")
need = {"transplanted__xlm15__omp_k64_rescaled", "transplanted__xlmr__omp_k64_rescaled"}
missing = need - set(built)
assert not missing, f"STOP - these are missing: {missing}"
print("\nBoth swapped models are ready. Continue to Step 8.")

---## Step 8 — 🚦 FIRST DECISION POINT: is the swapped model still alive?**Do this before the 3–4 hour training run.** It takes ~10 minutes and cansave you the whole afternoon.Swapping a model's tokenizer is drastic surgery. This checks whether thepatient survived, by covering up words and asking the model to guess them(this is what these models are originally trained to do).- **Original model:** we know xlmr is good at this and xlm15 is not.- **After the swap:** if the score collapses to **zero**, the surgery  destroyed the model's language ability, and any improvement you see later  is not "better tokenization" — it is a differently-broken model.This is measured on ~300 real Azerbaijani sentences (~1,300 covered words),which is enough to tell a working model from a dead one.

In [ ]:
!python -m src.transplant.top1_accuracy --config configs/pilot_t4.yaml --device cuda

In [ ]:
import json
r = json.load(open("results_pilot/top1_accuracy.json", encoding="utf-8"))
prov = r.get("eval_text_provenance", {})
print(f"Measured on: {prov.get('n_lines')} sentences, {prov.get('n_chars')} characters "
      f"(source: {prov.get('source')})\n")

verdicts = {}
for base in ("xlm15", "xlmr"):
    b, t = r[base]["base"], r[base]["transplanted"]
    print(f"{base}:")
    print(f"   original model : {b['n_correct']:4d} / {b['n_masked_tokens']:4d} correct"
          f"   ({100*b['top1_accuracy']:.1f}%)")
    if t is None:
        print("   swapped model  : NOT MEASURED"); continue
    print(f"   swapped model  : {t['n_correct']:4d} / {t['n_masked_tokens']:4d} correct"
          f"   ({100*t['top1_accuracy']:.1f}%)")
    verdicts[base] = (b["top1_accuracy"], t["top1_accuracy"])
    print()

print("=" * 66)
dead = [k for k, (bo, tr) in verdicts.items() if tr < 0.01]
alive = [k for k, (bo, tr) in verdicts.items() if tr >= 0.01]
if len(dead) == 2:
    print("RESULT: the swap destroyed BOTH models' language ability (~0% correct).")
    print()
    print("This is a genuine finding and worth reporting back NOW, before")
    print("training. It matters for two reasons:")
    print("  1. Any gain you see in Step 9 would not be 'better tokenization'.")
    print("  2. xlmr is our control. If its swap is broken too, a flat result")
    print("     there cannot tell 'nothing to fix' apart from 'we broke it'.")
    print()
    print("You can still run Step 9 - the numbers are informative either way -")
    print("but send this output to Shahin before spending 4 hours.")
elif dead:
    print(f"RESULT: the swap destroyed {dead} but {alive} survived.")
    print("Report this - an effect that appears only in the broken one is an")
    print("artifact of the surgery, not evidence about tokenization.")
else:
    print("RESULT: both swapped models still predict words. The surgery was")
    print("survivable, so a gain in Step 9 can honestly be read as a")
    print("tokenization effect. Continue to Step 9.")

---## Step 9 — The 12 training runs  ⏳ **3–4 hours**Twelve runs: 2 models × 2 choppers × 3 repeats. Each is ~15–20 minutes.**You can stop and resume.** The cell has a 3.5-hour budget and stopscleanly *between* runs — it will never be killed halfway through one. Everyfinished run is saved to your Drive, so re-running this cell later picks upexactly where it left off.While it runs you will see lines like `STEP step=120 loss=0.68 ...` — thatis normal progress. Watch that `loss` number: if it goes down, the model islearning.

In [ ]:
# Stops cleanly between runs after this many hours. Re-run the cell to continue.
BUDGET_HOURS = 3.5

!python -m src.training.run_grid --config configs/pilot_t4.yaml --phase 2 --streams 1 --budget-hours {BUDGET_HOURS}

In [ ]:
# How far along are we?
import json, glob
files = sorted(glob.glob("results_pilot/runs/*.json"))
print(f"{len(files)} of 12 runs finished\n")
for p in files:
    r = json.load(open(p, encoding="utf-8"))
    learned = "LEARNED " if r["escaped"] else "stuck   "
    print(f"  {r['base']:6s} {r['condition']:12s} seed={r['seed']:<5} {learned} "
          f"score={r['selected_validation_macro_f1']:.3f}  "
          f"({r['runtime_sec']/60:.0f} min)")
if len(files) < 12:
    print(f"\n{12-len(files)} still to go - re-run the cell above to continue.")
else:
    print("\nAll 12 done. Continue to Step 10.")

---## Step 10 — The results"Learned" below means the model actually learned to tell positive fromnegative. A model that got stuck predicting one answer for everything iscounted separately and is **not** averaged into the score — mixing those twothings together would hide exactly what we are trying to see.

In [ ]:
!python -m src.analysis.aggregate --config configs/pilot_t4.yaml
!python -m src.analysis.stats     --config configs/pilot_t4.yaml
!python -m src.analysis.decompose --config configs/pilot_t4.yaml
!python -m src.analysis.report    --config configs/pilot_t4.yaml

In [ ]:
import json
cells = {(c["base"], c["condition"]): c for c in
         json.load(open("results_pilot/decompose.json", encoding="utf-8"))["cells"]}

def show(base, label):
    a, b = cells[(base, "baza")], cells[(base, "tokenizator")]
    print(f"{label}")
    print(f"   original chopper : learned {a['escape_rate_count']}   score "
          f"{a['conditional_macro_f1'] if a['conditional_macro_f1'] is None else round(a['conditional_macro_f1'],3)}")
    print(f"   swapped  chopper : learned {b['escape_rate_count']}   score "
          f"{b['conditional_macro_f1'] if b['conditional_macro_f1'] is None else round(b['conditional_macro_f1'],3)}")
    print()
    return a, b

print("=" * 66)
x_a, x_b = show("xlm15", "xlm15  (NEVER saw Azerbaijani - the one we expect to improve)")
r_a, r_b = show("xlmr",  "xlmr   (ALREADY knows Azerbaijani - the control, expect no change)")

---## Step 11 — 🚦 The verdictThis cell reads the four boxes above and tells you what they mean for thefull experiment. The rules were written **before** any data was collected,so the answer cannot be talked into being whatever we hoped for.

In [ ]:
import json
cells = {(c["base"], c["condition"]): c for c in
         json.load(open("results_pilot/decompose.json", encoding="utf-8"))["cells"]}

def esc(b, c): return cells[(b, c)]["n_escaped"]
def f1(b, c):  return cells[(b, c)]["conditional_macro_f1"]

xb, xt = esc("xlm15", "baza"), esc("xlm15", "tokenizator")
rb, rt = esc("xlmr", "baza"),  esc("xlmr", "tokenizator")
gain_x = None if f1("xlm15","baza") is None or f1("xlm15","tokenizator") is None \
         else f1("xlm15","tokenizator") - f1("xlm15","baza")
gain_r = None if f1("xlmr","baza") is None or f1("xlmr","tokenizator") is None \
         else f1("xlmr","tokenizator") - f1("xlmr","baza")

print("learned-counts (out of 3):")
print(f"   xlm15  original {xb}   swapped {xt}")
print(f"   xlmr   original {rb}   swapped {rt}")
print(f"score change from swapping:  xlm15 {gain_x}   xlmr {gain_r}")
print("=" * 66)

MEANINGFUL = 0.03   # smaller than this is inside seed-to-seed noise at 3 repeats

if xb + xt + rb + rt == 0:
    print("VERDICT 1 - NOTHING LEARNED ANYWHERE.  ==> STOP AND RE-PLAN")
    print("2000 training examples is below the threshold where this task is")
    print("learnable at all. This is a finding about detectability, not a bug.")
    print("Do NOT start the full grid and do NOT tune the optimizer. Move the")
    print("reference size up to 10000 first.")
elif rb + rt > 0 and xb + xt == 0:
    print("VERDICT 2 - ONLY xlmr LEARNED; xlm15 NEVER DID.  ==> RE-PLAN")
    print("The model the whole study is built around cannot do the task at this")
    print("data size, with or without the swap. The full grid would measure")
    print("nothing. Raise the data size before committing the GPU window.")
elif gain_x is not None and gain_r is not None and gain_x >= MEANINGFUL and gain_r >= MEANINGFUL:
    print("VERDICT 3 - THE SWAP HELPS *BOTH* MODELS.  ==> CONTROL FAILED")
    print("xlmr already chops Azerbaijani well, so it should NOT have gained.")
    print("That it did means the gain is coming from the surgery itself (a much")
    print("smaller vocabulary is simply easier to train), not from fixing a")
    print("tokenization deficit. The cross-model comparison cannot separate the")
    print("two. Report this - it is a real result - but the headline claim as")
    print("currently framed is not supported.")
elif gain_x is not None and gain_x <= -MEANINGFUL:
    print("VERDICT 4 - THE SWAP MAKES xlm15 WORSE.  ==> NEGATIVE RESULT")
    print("Consistent with Step 8 if the swapped model scored ~0 there. A")
    print("well-documented negative result is explicitly acceptable for this")
    print("project. Report it; do not tune until it turns positive.")
elif (gain_x is not None and gain_x >= MEANINGFUL and
      (gain_r is None or abs(gain_r) < MEANINGFUL)) or (xt - xb >= 2 and rt - rb <= 0):
    print("VERDICT 5 - THE PREDICTED PATTERN APPEARED.  ==> GO")
    print("The swap helps the model that never saw Azerbaijani and leaves the")
    print("one that did roughly alone. That is exactly the signature the")
    print("hypothesis predicts, and it is the hardest outcome to get by chance.")
    print("The full 114-run experiment is worth running.")
    print("Next: the A100 or vast.ai H100 notebook, starting with Tranche A.")
else:
    print("VERDICT 6 - MIXED / TOO NOISY TO CALL.")
    print("No clean pattern at 3 repeats. This is not a failure - 3 repeats can")
    print("only ever show 0, 1/3, 2/3 or 1, so a modest effect hides easily.")
    print("Send these numbers to Shahin. The likely next move is Tranche A on a")
    print("real GPU at the full 2000-step budget and 5 repeats, which is the")
    print("measurement this pilot is a cheap preview of.")

print()
print("-" * 66)
print("REMINDER: pilot settings (800 steps, 3 repeats) - NOT paper numbers.")
print("They live in results_pilot/ and can never mix into the real results.")

In [ ]:
from IPython.display import Image, display
for n in ("escape_rate.png", "conditional_macro_f1.png"):
    try:
        print(f"figures_pilot/{n}"); display(Image(f"figures_pilot/{n}"))
    except Exception as e:
        print(f"(no {n} yet: {e})")

---## Step 12 — Send the results backThis packages everything into one zip and downloads it to your computer.Send that file back.

In [ ]:
import shutil
shutil.make_archive("/content/pilot_results", "zip", "results_pilot")
print("Downloading pilot_results.zip - send this file back.")
from google.colab import files
files.download("/content/pilot_results.zip")

---## If something goes wrong| What you see | What to do ||---|---|| "NO GPU" | **Runtime → Change runtime type → T4 GPU → Save**, re-run Step 1 || Disconnected / "runtime restarted" | Re-run Steps 2, 3, 4, then the step you were on. Nothing is lost. || `CUDA out of memory` | **Runtime → Restart session**, then re-run Steps 2–5 and continue. Do **not** change any settings in the config files. || Step 5 shows failures | Stop. Send the output back — do not continue. || Step 7 seems frozen | It is CPU work with little output. Give it 2 hours before worrying. || Step 9 stopped early | Normal — it stops at its time budget. Just run the cell again. || `ModuleNotFoundError` | Re-run Step 4, then Step 2. |**One rule:** if a cell tells you to STOP, stop and send the output back.Every stop condition in this notebook was written down before the dataexisted, and each one means the result would be misleading if you pushedpast it.